In [ ]:
# -*- coding: utf-8 -*-
"""
LSTM 성능 평가 — 테스트 + 지표 출력
 
두 가지 입력 방식 지원:
  (A) 테스트 CSV (train과 다른 영상에서 추출한 lstm_test_data.csv)
  (B) 테스트 영상 폴더 (운동별 폴더, YOLO로 즉석 추출)
 
출력:
  - 시퀀스 단위 정확도 / 혼동행렬 / classification_report
  - 영상 단위 정확도 (영상의 시퀀스들을 다수결 → 실사용 관점)
 
⚠️ 테스트 데이터는 학습에 안 쓴 영상이어야 함! (프레임 누수 금지)
"""
 
import os
import glob
import json
import time
import numpy as np
import torch
import torch.nn as nn
from collections import Counter
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from config import BASE_DIR

In [ ]:
MODEL_PATH = "exercise_lstm_velo.pth"
CONFIG_PATH = "lstm_velo_config.json"

TEST_VIDEO_DIR = BASE_DIR / "lstm_test_video"
YOLO_WEIGHT = "yolov8s-pose.pt"

CAPTURE_SEC = 0.1
IMGSZ = 1280
DET_CONF = 0.4
SMOOTH_WIN = 3      # 좌표 이동평균 윈도우 (떨림 제거)
VEL_GAP = 5         # 속도 계산 프레임 간격 (떨림 상쇄)

def get_label_from_filename(fname):
    name = os.path.basename(fname).lower()

    if name.startswith("leg"):
        return 0
    elif name.startswith("lunge"):
        return 1
    elif name.startswith("plank"):
        return 2
    elif name.startswith("pushup") or name.startswith("knee_pushup"):
        return 3
    else:
        return None

In [180]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(CONFIG_PATH, encoding="utf-8") as f:
    CFG = json.load(f)

SEQ = CFG["sequence_length"]
INPUT_SIZE = CFG["input_size"]
CLASS_NAMES = CFG["class_names"]
USE_WEIGHT = CFG["use_weight"]
JOINT_WEIGHTS = np.array(CFG["joint_weights"], dtype=np.float32)


def coord_weight_vec():
    return np.repeat(JOINT_WEIGHTS, 2).astype(np.float32)


def smooth_coords(win34, k=SMOOTH_WIN):
    T = len(win34)
    if T < k or k <= 1:
        return win34.copy()
    out = win34.copy()
    half = k // 2
    for t in range(T):
        s = max(0, t - half); e = min(T, t + half + 1)
        out[t] = win34[s:e].mean(axis=0)
    return out
 
 

def make_features(win34, gap=VEL_GAP):
    """
    (T,34) → (T,71)
    좌표34 + 속도34 + 정적특징3
    정적특징:
      1) joint_var
      2) lower_var
      3) upper_speed_mean
    """
    win34 = np.asarray(win34, dtype=np.float32)

    if win34.shape[1] != 34:
        raise ValueError(f"win34는 (T,34)여야 합니다. 현재 shape: {win34.shape}")

    coords = smooth_coords(win34)
    T = len(coords)

    # 전체 관절 속도
    vel = np.zeros_like(coords)
    for t in range(T):
        prev = max(0, t - gap)
        vel[t] = coords[t] - coords[prev]

    # 전체 관절 분산
    joint_var = coords.var(axis=0).mean()

    # 하체 관절 분산
    lower_idx = []
    for j in [11, 12, 13, 14, 15, 16]:
        lower_idx += [j * 2, j * 2 + 1]

    lower_var = coords[:, lower_idx].var(axis=0).mean()

    # 상체 속도 평균: 어깨, 팔꿈치, 손목
    upper_idx = []
    for j in [5, 6, 7, 8, 9, 10]:
        upper_idx += [j * 2, j * 2 + 1]

    upper_vel = vel[:, upper_idx]                  # (T,12)
    upper_vel_xy = upper_vel.reshape(T, -1, 2)     # (T,6,2)
    upper_speed_mean = np.linalg.norm(
        upper_vel_xy,
        axis=2
    ).mean()

    static = np.tile(
        [joint_var, lower_var, upper_speed_mean],
        (T, 1)
    ).astype(np.float32)

    out = np.concatenate([win34, vel, static], axis=1)

    if out.shape[1] != INPUT_SIZE:
        raise ValueError(f"입력 차원 오류: {out.shape}, INPUT_SIZE={INPUT_SIZE}")

    return out
 
# 호환용 별칭 (기존 코드가 add_velocity 를 부르면 이걸 쓰도록)
def add_velocity(win34, **kwargs):
    return make_features(win34)

def make_seq(win34):
    w = add_velocity(win34)

    if USE_WEIGHT:
        cw = coord_weight_vec()

        full_w = np.concatenate([
            cw,
            cw,
            np.ones(3, dtype=np.float32)
        ])

        w = w * full_w

    return w

class ExerciseLSTM(nn.Module):
    def __init__(self):
        super().__init__()

        self.lstm = nn.LSTM(
            INPUT_SIZE,
            CFG["hidden_size"],
            CFG["num_layers"],
            batch_first=True,
            dropout=CFG["dropout"] if CFG["num_layers"] > 1 else 0.0
        )

        self.fc = nn.Linear(CFG["hidden_size"], CFG["num_classes"])

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


def seqs_from_videos(video_dir):
    from ultralytics import YOLO
    import cv2

    pose_model = YOLO(YOLO_WEIGHT)

    video_paths = []
    for ext in ["*.mp4", "*.mov", "*.avi", "*.m4v"]:
        video_paths.extend(glob.glob(os.path.join(video_dir, ext)))

    video_paths = sorted(video_paths)

    data = []

    for vp in video_paths:
        lab = get_label_from_filename(vp)

        if lab is None:
            print(f"[스킵] 라벨을 알 수 없는 파일명: {os.path.basename(vp)}")
            continue

        cap = cv2.VideoCapture(vp)

        if not cap.isOpened():
            print(f"[스킵] 영상을 열 수 없음: {os.path.basename(vp)}")
            continue

        fps = cap.get(cv2.CAP_PROP_FPS) or 30
        interval = max(1, int(fps * CAPTURE_SEC))

        coords = []
        frame_count = 0
        processed_frames = 0

        start_time = time.time()

        while True:
            ok, frame = cap.read()

            if not ok:
                break

            if frame_count % interval == 0:
                result = pose_model.predict(
                    frame,
                    imgsz=IMGSZ,
                    conf=DET_CONF,
                    verbose=False
                )[0]

                if result.keypoints is not None and result.keypoints.xyn is not None:
                    if len(result.keypoints.xyn) > 0:
                        kp = result.keypoints.xyn[0].cpu().numpy()
                        coords.append(kp.flatten().astype(np.float32))
                        processed_frames += 1

            frame_count += 1

        cap.release()

        elapsed = time.time() - start_time
        video_fps = frame_count / elapsed if elapsed > 0 else 0

        coords = np.array(coords, dtype=np.float32)

        if len(coords) < SEQ:
            print(
                f"[스킵] {os.path.basename(vp)}: "
                f"유효 관절 프레임 {len(coords)}개 < {SEQ}"
            )
            continue

        seqs = [
            make_seq(coords[i:i + SEQ])
            for i in range(0, len(coords) - SEQ + 1, 1)
        ]

        data.append({
            "name": os.path.basename(vp),
            "seqs": seqs,
            "label": lab,
            "frame_count": frame_count,
            "processed_frames": processed_frames,
            "elapsed": elapsed,
            "fps": video_fps,
        })

    return data

In [181]:

def main():
    print(f"장치: {device}")

    model = ExerciseLSTM().to(device)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()

    data = seqs_from_videos(TEST_VIDEO_DIR)

    if not data:
        print("[오류] 테스트 데이터 없음. 경로/파일명/영상 길이 확인.")
        return

    seq_true = []
    seq_pred = []

    vid_true = []
    vid_pred = []

    all_video_fps = []
    all_elapsed = []
    all_lstm_latency = []

    print("=" * 70)

    for item in data:
        name = item["name"]
        seqs = item["seqs"]
        lab = item["label"]

        X = torch.tensor(np.array(seqs), dtype=torch.float32).to(device)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        lstm_start = time.time()

        with torch.no_grad():
            logits = model(X)
            preds = logits.argmax(1).cpu().numpy()

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        lstm_elapsed = time.time() - lstm_start
        lstm_latency_ms = (lstm_elapsed / len(seqs)) * 1000

        seq_true += [lab] * len(preds)
        seq_pred += preds.tolist()

        vid_p = Counter(preds).most_common(1)[0][0]

        vid_true.append(lab)
        vid_pred.append(vid_p)

        all_video_fps.append(item["fps"])
        all_elapsed.append(item["elapsed"])
        all_lstm_latency.append(lstm_latency_ms)

        mark = "O" if vid_p == lab else "X"

        print(
            f"[{mark}] {name} | "
            f"정답: {CLASS_NAMES[lab]} | "
            f"예측: {CLASS_NAMES[vid_p]} | "
            f"시퀀스: {len(preds)}개 | "
            f"처리시간: {item['elapsed']:.2f}s | "
            f"FPS: {item['fps']:.2f} | "
            f"LSTM Latency: {lstm_latency_ms:.3f} ms/seq"
        )

    print("=" * 70)

    print(f"\n■ 영상 단위 Accuracy: {accuracy_score(vid_true, vid_pred):.4f}")
    print(f"■ 시퀀스 단위 Accuracy: {accuracy_score(seq_true, seq_pred):.4f}")

    print(f"\n■ 평균 영상 처리 시간: {np.mean(all_elapsed):.2f}s")
    print(f"■ 평균 처리 FPS(YOLO+전처리 포함): {np.mean(all_video_fps):.2f}")
    print(f"■ 평균 LSTM Latency: {np.mean(all_lstm_latency):.3f} ms/seq")

    labels = list(range(len(CLASS_NAMES)))

    print("\n[시퀀스 단위 classification_report]")
    print(
        classification_report(
            seq_true,
            seq_pred,
            labels=labels,
            target_names=CLASS_NAMES,
            zero_division=0
        )
    )

    print("[혼동 행렬] 행=정답, 열=예측")
    cm = confusion_matrix(seq_true, seq_pred, labels=labels)

    header = "        " + " ".join(f"{c[:4]:>6}" for c in CLASS_NAMES)
    print(header)

    for i, row in enumerate(cm):
        print(f"{CLASS_NAMES[i][:6]:>6} " + " ".join(f"{v:>6}" for v in row))

In [182]:
if __name__ == "__main__":
    main()

장치: cuda
[O] leg_01.mp4 | 정답: 레그레이즈 | 예측: 레그레이즈 | 시퀀스: 78개 | 처리시간: 2.73s | FPS: 81.70 | LSTM Latency: 0.014 ms/seq
[O] leg_02.mp4 | 정답: 레그레이즈 | 예측: 레그레이즈 | 시퀀스: 55개 | 처리시간: 2.24s | FPS: 98.89 | LSTM Latency: 0.000 ms/seq
[X] leg_03.mp4 | 정답: 레그레이즈 | 예측: 플랭크 | 시퀀스: 8개 | 처리시간: 1.65s | FPS: 75.36 | LSTM Latency: 0.125 ms/seq
[X] leg_04.mp4 | 정답: 레그레이즈 | 예측: 플랭크 | 시퀀스: 78개 | 처리시간: 4.53s | FPS: 75.47 | LSTM Latency: 0.013 ms/seq
[O] lunge_01.mp4 | 정답: 런지 | 예측: 런지 | 시퀀스: 304개 | 처리시간: 28.44s | FPS: 23.73 | LSTM Latency: 0.003 ms/seq
[O] lunge_02.mp4 | 정답: 런지 | 예측: 런지 | 시퀀스: 54개 | 처리시간: 2.60s | FPS: 67.58 | LSTM Latency: 0.000 ms/seq
[O] lunge_03.mp4 | 정답: 런지 | 예측: 런지 | 시퀀스: 259개 | 처리시간: 21.24s | FPS: 27.59 | LSTM Latency: 0.004 ms/seq
[O] lunge_05.mp4 | 정답: 런지 | 예측: 런지 | 시퀀스: 140개 | 처리시간: 5.95s | FPS: 58.31 | LSTM Latency: 0.007 ms/seq
[O] lunge_06.mp4 | 정답: 런지 | 예측: 런지 | 시퀀스: 38개 | 처리시간: 2.71s | FPS: 79.73 | LSTM Latency: 0.026 ms/seq
[O] plank_01.mp4 | 정답: 플랭크 | 예측: 플랭크 | 시퀀스: 144개 | 처리시간: 